In [1]:
from typing import List, Any
from langchain_core.documents import Document
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

c:\Users\user\Desktop\contractAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def load_all_document(data_dir:str)->List[Any]:
    """Load all documents from data directory"""
    data_path = Path(data_dir).resolve()
    print(f"[DEBUG] data path: {data_path}")
    documents = []
    ## PDFs
    pdf_files = list(data_path.glob("**/*.pdf"))
    print(f"[DEBUG] Found {len(pdf_files)} pdf files")

    for pdf_file in pdf_files:
        try:
            print(f"[DEBUG] Loading pdf: {pdf_file}")
            loader = PyMuPDFLoader(str(pdf_file))
            docs = loader.load()
            for doc in docs:
                doc.metadata["file_type"] = "pdf"
                doc.metadata['file_name'] = pdf_file.name
                # print(doc.metadata)
            documents.extend(docs)
            print(f"[DEBUG] loaded document: {pdf_file}")
        except Exception as e:
            print(f"[ERROR] Failed to load {pdf_file}: {e}")
    print(f"loaded {len(documents)}")
    return documents


In [3]:
all_documents = load_all_document("../data")

[DEBUG] data path: C:\Users\user\Desktop\contractAssistant\data
[DEBUG] Found 2 pdf files
[DEBUG] Loading pdf: C:\Users\user\Desktop\contractAssistant\data\pdf\ADUROBIOTECH,INC_06_02_2020-EX-10.7-CONSULTING AGREEMENT(1).PDF
[DEBUG] loaded document: C:\Users\user\Desktop\contractAssistant\data\pdf\ADUROBIOTECH,INC_06_02_2020-EX-10.7-CONSULTING AGREEMENT(1).PDF
[DEBUG] Loading pdf: C:\Users\user\Desktop\contractAssistant\data\pdf\CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF
[DEBUG] loaded document: C:\Users\user\Desktop\contractAssistant\data\pdf\CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT.PDF
loaded 15


In [ ]:
## meata data extraction
import re


In [8]:

CONTRACT_TYPE_PATTERNS = {
    "NDA": [
        r"non.disclosure\s+agreement",
        r"confidentiality\s+agreement",
        r"\bNDA\b",
    ],
    "MSA": [
        r"master\s+service[s]?\s+agreement",
        r"master\s+agreement",
        r"\bMSA\b",
    ],
    "SOW": [
        r"statement\s+of\s+work",
        r"\bSOW\b",
        r"scope\s+of\s+work",
    ],
    "SLA": [
        r"service\s+level\s+agreement",
        r"\bSLA\b",
        r"service\s+level\s+objective",
    ],
    "RENEWAL": [
        r"renewal\s+agreement",
        r"contract\s+renewal",
        r"amendment\s+and\s+renewal",
    ],
}
# Covers formats: January 1, 2023 / Jan 1, 2023 / 01/01/2023 / 2023-01-01
DATE_PATTERN = re.compile(
    r"""
    (?:
        (?:January|February|March|April|May|June|July|August|
           September|October|November|December|
           Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)
        [\s,]+\d{1,2}[\s,]+\d{4}
    )
    |
    (?:\d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4})
    |
    (?:\d{4}[\/\-]\d{1,2}[\/\-]\d{1,2})
    """,
    re.VERBOSE | re.IGNORECASE,
)

EXPIRY_CONTEXT = re.compile(
    r"(?:expir|terminat|end\s+date|renewal\s+date|expire[sd]?|expiration).*?"
    r"(" + DATE_PATTERN.pattern + r")",
    re.IGNORECASE | re.VERBOSE | re.DOTALL,
)

EFFECTIVE_CONTEXT = re.compile(
    r"(?:effective\s+date|commenc|start\s+date|dated\s+as\s+of|entered\s+into).*?"
    r"(" + DATE_PATTERN.pattern + r")",
    re.IGNORECASE | re.VERBOSE | re.DOTALL,
)

def detecte_contract_type(text):
    text_lower = text.lower()
    for contract_type, patterns in CONTRACT_TYPE_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern,text_lower,re.IGNORECASE):
                return contract_type
    return "UKNOWN"

def extract_dates(text: str) -> dict:
    dates = {}

    match = EFFECTIVE_CONTEXT.search(text[:3000])   # usually in header/preamble
    if match:
        dates["effective_date"] = match.group(1).strip()

    match = EXPIRY_CONTEXT.search(text)
    if match:
        dates["expiry_date"] = match.group(1).strip()

    return dates
def enrich_metadata(documents):
    for doc in documents:
        text = doc.page_content
        contract_type = detecte_contract_type(text)
        contract_date = extract_dates(text)

        doc.metadata['contract_type'] = contract_type
        doc.metadata.update(contract_date)
    return documents
    


In [12]:
print(enrich_metadata(all_documents)[1])

page_content='(b) Consultant shall keep all Confidential Information confidential, and Consultant shall not disclose, disseminate, publish, reproduce or use
Confidential Information except to perform the Services. If Consultant is required by judicial or administrative process to disclose Confidential
Information, Consultant shall promptly notify Aduro to allow Aduro a reasonable time to oppose such process and Consultant shall reasonably
cooperate in Aduro’s efforts.
(c) On Aduro’s request, or upon the termination or expiration of this Agreement, Consultant shall immediately: (i) stop using Confidential
Information; (ii) return all materials provided by Aduro to Consultant that contain Confidential Information, except for one copy that may be retained by
Consultant’s legal counsel to confirm compliance with the obligations under this Agreement; (iii) destroy all copies of Confidential Information in any
form including Confidential Information contained in computer memory or data stora

### Chunking

In [ ]:
import re
from typing import List
# from langchain.schema import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings


SECTION_PATTERN = re.compile(r"\n\s*(\d+)\.\s+[A-Z][^\n]+")


def split_by_sections(text: str):
    """
    Split contract text by legal section headers like:
    1. Services
    2. Compensation
    """

    matches = list(SECTION_PATTERN.finditer(text))

    if not matches:
        return [text]

    sections = []

    for i, match in enumerate(matches):
        start = match.start()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)

        sections.append(text[start:end].strip())

    return sections


def chunk_contract_documents(documents: List[Document]) -> List[Document]:
    """
    Chunk contracts using section-aware chunking,
    then semantic chunking for large sections.
    """

    embeddings = OpenAIEmbeddings()

    semantic_chunker = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=95,
    )

    all_chunks = []

    for doc in documents:

        text = doc.page_content

        # split by contract sections
        sections = split_by_sections(text)

        for section in sections:

            if len(section) < 2000:
                # small section → keep as one chunk
                chunk = Document(
                    page_content=section,
                    metadata=doc.metadata.copy()
                )

                all_chunks.append(chunk)

            else:
                # large section → semantic chunking
                semantic_chunks = semantic_chunker.create_documents(
                    [section],
                    [doc.metadata]
                )

                all_chunks.extend(semantic_chunks)

    print(f"[INFO] Created {len(all_chunks)} chunks")

    return all_chunks

In [ ]:
### splitting text into chunks
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

In [ ]:
## a pattern that designed to match numbered list items that start on a new line
# The re.compile() function converts this string into a pattern object, which is more efficient if you plan to use the regex multiple times in your code

SECTION_PATTERN = re.compile(r"\n\s*(\d+)\.\s+[A-Z][^\n]+")

def split_by_sections(text):
    """split contract text by legal sections heading"""
    matches = list(SECTION_PATTERN.finditer(text))

    if not matches:
        return [text]
    sections = []
    for i, match in enumerate(matches):
        start = match.start()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)
        # - Starts at the **current heading position**
        # - Ends at the **next heading's position** (or end of text for the last section)
        sections.append(text[start:end].strip())

    return sections

def chunk_contract_document(documents):
    """
    Chunk contracts using section-aware chunking,
    then semantic chunking for large sections.
    """
    embeddings = OpenAIEmbeddings()
    semantic_chunker = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=95,
    )
    all_chunks = []
    
    for doc in documents:

        sections = split_by_sections(doc.page_content)
        for section in sections:
            if len(section) < 2000:
                chunk = Document(page_content=section,
                                 metadata = doc.metadata.copy())
                all_chunks.append(chunk)
            else:
                chunk = semantic_chunker.create_documents(
                    [section],
                    [doc.metadata]
                )
                all_chunks.append(chunk)
    return all_chunks

### Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbbeddingPipeline:
    def __init__(self, model_name:str ="BAAI/bge-large-en-v1.5"):
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)


    def embedde_chunks(self,  chunks):
        print(f"generating embedding {len(chunks)} chunks ...")
        texts = [chunk.page_content for chunk in chunks]
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"embeddings shape: {embeddings.shape}")
        return embeddings

    
    # 9:48

In [ ]:
print("hel")

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-large-en-v1.5")

print("Model device:", model.device)

ImportError: cannot import name 'PreTrainedModel' from 'transformers' (c:\Users\elbou\Desktop\contractAssistant\contractAssistant\.venv\Lib\site-packages\transformers\__init__.py)

In [4]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2.10.0+cpu
CUDA available: False
